# From LLM to Omni: any-to-any shapes

## Learning goals
- Distinguish multimodal *input* from any-to-any *output*
- Recognize the three model families vLLM-Omni serves
- See why heterogeneous outputs force a pipeline, not a single decoder

> **Mac track.** Every executed cell below is a pure-Python simulation that runs on any Mac with no GPU. Cells labelled **Linux GPU lab** show the *real* vLLM-Omni commands — they are shown as text and are **not** run here, because the official `vllm serve --omni` runtime is CUDA-only.

## Input vs output modality

A plain LLM is text->text. A *multimodal-input* model is (text|image|audio)->text. An *any-to-any* model can also emit non-text: text->image, audio->audio, image->image.

```text
text-in            -> [LLM]        -> text-out
text+image+audio   -> [Omni model] -> text | image | audio
```

Emitting audio or pixels is not one more decoding step — it needs a *different kind* of stage (an audio codec, a diffusion decoder). That is the whole reason vLLM-Omni exists.

In [ ]:
families = {
    'Omni-modality': ['Qwen3-Omni', 'BAGEL', 'Cosmos', 'HunyuanImage'],
    'TTS':           ['Qwen3-TTS', 'CosyVoice3', 'VoxCPM2'],
    'Diffusion':     ['Qwen-Image', 'Wan2.2', 'FLUX'],
}
for family, models in families.items():
    print(f'{family:14}: ' + ', '.join(models))

## The output shape drives the architecture

Below we tag a few example models with their input and output modalities. Notice that the moment an output is `image` or `audio`, the runtime needs a non-AR stage to produce it.

In [ ]:
examples = [
    ('Qwen3-Omni', 'text+image+audio', 'text+audio'),
    ('FLUX',       'text',             'image'),
    ('Qwen3-TTS',  'text',             'audio'),
]
for model, inp, out in examples:
    needs_nonar = any(m in out for m in ('image', 'audio'))
    print(f'{model:12} {inp:18} -> {out:12} | non-AR stage needed: {needs_nonar}')

## Exercise

A product needs speech-in, speech-out translation. Which family, and how many *kinds* of stage (AR text, AR audio-codec, codec-to-wave) does the output path imply? Write your guess, then run the solution.

In [ ]:
# Solution
print('Family: Omni-modality (understands audio, emits audio).')
print('Output path stages: Thinker (AR text) -> Talker (AR audio codes) -> Code2Wav.')
print('That is the exact Qwen3-Omni speech pipeline you meet in notebook 03.')

## Source lab

Read the supported-model tables in the repo `README.md`. For one model in each family, find its config under `vllm_omni/deploy/` and note how many stages it declares.